# Kaggle 01 — Tiền xử lý dữ liệu

Notebook mẫu để chạy pipeline trên Kaggle. Gắn raw dataset private vào `/kaggle/input`; output ghi vào `/kaggle/working`. Không có dữ liệu thật trong repo, nên hãy điền metadata thật trước khi chạy xử lý.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/<owner>/<repo>.git"  # thay bằng URL repo
REPO_DIR = Path('/kaggle/working/driver-behavior-monitoring')
RAW_ROOT = Path('/kaggle/input/<raw-dataset-slug>')  # dataset private đã gắn
WORK_ROOT = Path('/kaggle/working/dbd_work')
os.environ['DBM_RAW_ROOT'] = str(RAW_ROOT)
os.environ['DBM_WORK_ROOT'] = str(WORK_ROOT)
if not REPO_DIR.exists():
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        pass
    clone_url = REPO_URL.replace('https://', f'https://{token}@') if token and '<owner>' not in REPO_URL else REPO_URL
    subprocess.run(['git', 'clone', clone_url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repo:', REPO_DIR, 'raw mounted:', RAW_ROOT.exists())

In [ ]:
# Kaggle thường đã có torch/numpy. Cài package project trước; cài dependency bổ sung khi Internet bật.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps', '-q'], check=True)
# Nếu Internet tắt, attach wheel dataset và cài wheel từ /kaggle/input/<wheel-dataset> thay cho lệnh dưới.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
import torch
print('Python:', sys.version, '| Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Kiểm kê và chuẩn bị

Khởi tạo thư mục repo nếu cần. Điền metadata thật trong `annotations/metadata.xlsx`; các dòng ví dụ trong template không phải dữ liệu huấn luyện.

In [ ]:
subprocess.run([sys.executable, 'driver_behavior_dataset_v01/scripts/init_dataset.py', '--help'], check=True)
subprocess.run([sys.executable, 'driver_behavior_dataset_v01/scripts/inventory.py', '--help'], check=True)
subprocess.run([sys.executable, 'driver_behavior_dataset_v01/scripts/make_metadata_template.py', '--help'], check=True)
subprocess.run([sys.executable, 'driver_behavior_dataset_v01/scripts/validate_metadata.py', '--help'], check=True)

In [ ]:
# Các lệnh thực thi khi đã cấu hình và kiểm tra dữ liệu thật. Bỏ comment theo từng bước.
commands = [
    ['driver_behavior_dataset_v01/scripts/init_dataset.py'],
    ['driver_behavior_dataset_v01/scripts/inventory.py'],
    ['driver_behavior_dataset_v01/scripts/validate_metadata.py'],
    ['driver_behavior_dataset_v01/scripts/cut_segments.py'],
    ['driver_behavior_dataset_v01/scripts/make_splits.py'],
    ['driver_behavior_dataset_v01/scripts/extract_frames.py'],
    ['driver_behavior_dataset_v01/scripts/extract_landmarks.py'],
]
for command in commands:
    print(' '.join(command) + ' --help  # review arguments before running')

## Quy trình chạy thật

Sau khi kiểm tra metadata và mount dataset, chạy lần lượt `inventory.py`, `validate_metadata.py`, `cut_segments.py`, `make_splits.py`, `extract_frames.py` và `extract_landmarks.py`. Dùng `--overwrite` chỉ khi chủ động muốn tạo lại artifact. File `face_landmarker.task` có thể mount từ Kaggle Dataset và đặt `DBM_FACE_LANDMARKER_PATH`. Kiểm tra ảnh bằng `visualize_landmarks.py` trước khi huấn luyện.